# 05: Generate Your Own Lehmer Pair Predictions

Use the Monster resonance formula to predict where new extreme Lehmer pairs should appear.

$$\gamma^* = k \cdot \lambda_M \cdot \frac{\log p_i}{\log p_j}$$

Choose any two Monster primes and a range of $k$ values to generate testable predictions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math

LAMBDA_M = 19.76
MONSTER_PRIMES = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 41, 47, 59, 71]

print('Monster primes:', MONSTER_PRIMES)
print(f'Monster wavelength: lambda_M = {LAMBDA_M}')

## Generate Predictions for a Specific Prime Pair

In [ ]:
def generate_predictions(p_i, p_j, k_min=1, k_max=1000, gamma_min=1000, gamma_max=1e10):
    """Generate Lehmer pair predictions for a specific Monster prime pair."""
    if p_i == p_j:
        raise ValueError('p_i and p_j must be different')
    if p_i not in MONSTER_PRIMES or p_j not in MONSTER_PRIMES:
        raise ValueError('Both primes must be Monster primes')
    
    ratio = math.log(p_i) / math.log(p_j)
    predictions = []
    
    for k in range(k_min, k_max + 1):
        gamma = k * LAMBDA_M * ratio
        if gamma_min <= gamma <= gamma_max:
            predictions.append({
                'gamma_predicted': round(gamma, 4),
                'k': k,
                'prime_i': p_i,
                'prime_j': p_j,
                'formula': f'log{p_i}/log{p_j}',
                'log_ratio': round(ratio, 6),
            })
    
    return pd.DataFrame(predictions)

# Example: predictions for log(2)/log(17) (same ratio as known Pair #2)
preds = generate_predictions(2, 17, k_min=3000, k_max=4000)
print(f'Predictions for log(2)/log(17), k in [3000, 4000]:')
print(f'Total: {len(preds)} predictions')
print(f'\nFirst 10:')
print(preds.head(10).to_string(index=False))

# Check: known pair #2 has k=3547, gamma=17143.79
known_k = 3547
known_gamma = 17143.79
our_pred = preds[preds['k'] == known_k]
if len(our_pred) > 0:
    print(f'\nKnown pair #2: k={known_k}, actual gamma={known_gamma}')
    print(f'Our prediction: gamma={our_pred.iloc[0]["gamma_predicted"]}')
    print(f'Error: {abs(known_gamma - our_pred.iloc[0]["gamma_predicted"]) / known_gamma * 100:.4f}%')

## Scan All Monster Prime Pairs

In [ ]:
def scan_all_pairs(gamma_target, tolerance=0.1):
    """Find which Monster prime pairs and k values predict gamma near a target."""
    results = []
    gamma_low = gamma_target * (1 - tolerance)
    gamma_high = gamma_target * (1 + tolerance)
    
    for p_i in MONSTER_PRIMES:
        for p_j in MONSTER_PRIMES:
            if p_i == p_j:
                continue
            ratio = math.log(p_i) / math.log(p_j)
            step = LAMBDA_M * ratio
            if step <= 0:
                continue
            
            k_low = max(1, int(gamma_low / step))
            k_high = int(gamma_high / step) + 1
            
            for k in range(k_low, k_high + 1):
                gamma = k * step
                if gamma_low <= gamma <= gamma_high:
                    error = abs(gamma - gamma_target) / gamma_target * 100
                    results.append({
                        'k': k,
                        'p_i': p_i,
                        'p_j': p_j,
                        'gamma_predicted': round(gamma, 4),
                        'error_pct': round(error, 6),
                    })
    
    df = pd.DataFrame(results).sort_values('error_pct')
    return df

# Find what predicts the known Lehmer pair at gamma = 7005.06
matches = scan_all_pairs(7005.06, tolerance=0.01)
print(f'Predictions near gamma = 7005.06 (+/- 1%):')
print(f'Found {len(matches)} matches')
if len(matches) > 0:
    print(matches.head(10).to_string(index=False))

## Prediction Density Map

Where are the predictions densest? This shows the "fingerprint" of Monster resonance.

In [ ]:
# Generate predictions from all pairs for k=1..200
all_gammas = []
for p_i in MONSTER_PRIMES:
    for p_j in MONSTER_PRIMES:
        if p_i == p_j:
            continue
        ratio = math.log(p_i) / math.log(p_j)
        for k in range(1, 201):
            gamma = k * LAMBDA_M * ratio
            if 10 < gamma < 10000:
                all_gammas.append(gamma)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Top: Histogram
ax1 = axes[0]
ax1.hist(all_gammas, bins=200, color='blue', alpha=0.7, edgecolor='none')
# Mark known Lehmer pair
ax1.axvline(x=7005.06, color='red', linewidth=2, linestyle='--', label='Known Lehmer pair (gamma=7005.06)')
ax1.set_xlabel('Predicted gamma')
ax1.set_ylabel('Prediction count')
ax1.set_title('Density of Monster Resonance Predictions (k=1..200)', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Bottom: Scatter by prime pair
ax2 = axes[1]
pair_idx = 0
for p_i in MONSTER_PRIMES[:5]:  # First 5 primes for clarity
    for p_j in MONSTER_PRIMES[:5]:
        if p_i >= p_j:
            continue
        ratio = math.log(p_i) / math.log(p_j)
        gammas_pair = [k * LAMBDA_M * ratio for k in range(1, 101)]
        gammas_pair = [g for g in gammas_pair if 10 < g < 5000]
        ax2.scatter(gammas_pair, [pair_idx]*len(gammas_pair), s=5, alpha=0.5)
        ax2.text(-200, pair_idx, f'{p_i}/{p_j}', fontsize=8, ha='right', va='center')
        pair_idx += 1

ax2.axvline(x=7005.06, color='red', linewidth=2, linestyle='--')
ax2.set_xlabel('Predicted gamma')
ax2.set_ylabel('Prime pair index')
ax2.set_title('Predictions by Prime Pair (first 5 Monster primes)', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Total predictions in [10, 10000]: {len(all_gammas):,}')

## Export Custom Predictions

Generate and save your own prediction set.

In [ ]:
# Choose your parameters
P_I = 7    # Choose a Monster prime
P_J = 23   # Choose another Monster prime  
K_MIN = 1
K_MAX = 500000
GAMMA_MIN = 1e6
GAMMA_MAX = 1e9

custom_preds = generate_predictions(P_I, P_J, K_MIN, K_MAX, GAMMA_MIN, GAMMA_MAX)
print(f'Generated {len(custom_preds):,} predictions for log({P_I})/log({P_J})')
print(f'Gamma range: [{custom_preds["gamma_predicted"].min():.0f}, {custom_preds["gamma_predicted"].max():.0f}]')

if len(custom_preds) > 0:
    print(f'\nSample (first 10):')
    print(custom_preds.head(10).to_string(index=False))
    
    # Uncomment to save:
    # custom_preds.to_csv('my_predictions.csv', index=False)
    # print('Saved to my_predictions.csv')